# Predicting diabetes from age, BMI, income and exercise (CDC BRFSS 2025)

**Question:** Can four simple facts predict who has diabetes, and does gradient boosting beat a simple baseline?

**Data:** 268,563 people with complete answers (out of 356,158). 14.7% have diabetes. Split 80/20 into training and test sets, random seed 42, the same split for both models.

**Results (test set, 53,713 people):**
- AUC: logistic regression 0.752, gradient boosting 0.762.
- At the default 0.5 cutoff, both models catch very few cases (recall 4.3% and 1.1%), so accuracy (85.3%) is no better than always guessing "no diabetes".
- At a 0.2 cutoff, gradient boosting catches 61% of cases, and about 30% of the people it flags have diabetes (the overall rate is 14.7%).
- Most useful clues: age, then BMI, income, exercise.

**Limitations:** shows prediction, not cause. I compared cutoffs on the test set, so a separate validation split would be stricter. Age and income were turned into numbers (band midpoints and income order). The data is self-reported and not weighted to the US population. Only four clues were used.

In [1]:
import pandas as pd

df = pd.read_csv("data/brfss_clean.csv.gz")
print(df.shape)
print(df.columns.tolist())

(356158, 14)
['state', 'sex', 'age_group', 'income', 'bmi', 'smoked_100', 'exercise', 'general_health', 'diabetes', 'heart_disease', 'heart_attack', 'has_diabetes', 'age_band', 'bmi_group']


In [2]:
cols = ["age_group", "bmi", "income", "exercise", "has_diabetes"]
model_df = df[cols].dropna()

print("Before:", len(df))
print("After:", len(model_df))
print("Removed:", len(df) - len(model_df))
print()
print(model_df["age_group"].unique())
print(model_df["has_diabetes"].value_counts())

Before: 356158
After: 268563
Removed: 87595

<ArrowStringArray>
['70-74', '60-64',   '80+', '75-79', '65-69', '55-59', '45-49', '30-34',
 '50-54', '40-44', '35-39', '18-24', '25-29']
Length: 13, dtype: str
has_diabetes
0.0    229063
1.0     39500
Name: count, dtype: int64


In [3]:
age_numbers = {"18-24": 21, "25-29": 27, "30-34": 32, "35-39": 37,
               "40-44": 42, "45-49": 47, "50-54": 52, "55-59": 57,
               "60-64": 62, "65-69": 67, "70-74": 72, "75-79": 77, "80+": 82}

income_order = ["<$10k", "$10-15k", "$15-20k", "$20-25k", "$25-35k",
                "$35-50k", "$50-75k", "$75-100k", "$100-150k", "$150-200k", "$200k+"]
income_numbers = {name: i + 1 for i, name in enumerate(income_order)}

model_df = model_df.copy()
model_df["age"] = model_df["age_group"].map(age_numbers)
model_df["income_level"] = model_df["income"].map(income_numbers)
model_df["exercise_yes"] = (model_df["exercise"] == "Yes").astype(int)

print(model_df[["age", "bmi", "income_level", "exercise_yes", "has_diabetes"]].isna().sum())
print()
print(model_df[["age", "bmi", "income_level", "exercise_yes"]].describe().round(1))

age             0
bmi             0
income_level    0
exercise_yes    0
has_diabetes    0
dtype: int64

            age       bmi  income_level  exercise_yes
count  268563.0  268563.0      268563.0      268563.0
mean       55.7      28.6           7.1           0.8
std        18.2       6.5           2.4           0.4
min        21.0      12.0           1.0           0.0
25%        42.0      24.2           5.0           1.0
50%        57.0      27.5           7.0           1.0
75%        72.0      31.9           9.0           1.0
max        82.0      98.4          11.0           1.0


In [4]:
from sklearn.model_selection import train_test_split

X = model_df[["age", "bmi", "income_level", "exercise_yes"]]
y = model_df["has_diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training people:", len(X_train))
print("Test people:", len(X_test))
print()
print("Diabetes rate in training:", round(y_train.mean() * 100, 1), "%")
print("Diabetes rate in test:", round(y_test.mean() * 100, 1), "%")

Training people: 214850
Test people: 53713

Diabetes rate in training: 14.7 %
Diabetes rate in test: 14.7 %


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score

baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
baseline.fit(X_train, y_train)

pred = baseline.predict(X_test)
prob = baseline.predict_proba(X_test)[:, 1]

print("Always guess 'no diabetes':", round((1 - y_test.mean()) * 100, 1), "% correct")
print("Model accuracy:", round(accuracy_score(y_test, pred) * 100, 1), "%")
print("AUC:", round(roc_auc_score(y_test, prob), 3))
print("Precision:", round(precision_score(y_test, pred), 3))
print("Recall:", round(recall_score(y_test, pred), 3))

Always guess 'no diabetes': 85.3 % correct
Model accuracy: 85.0 %
AUC: 0.752
Precision: 0.418
Recall: 0.043


In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier

boosted = HistGradientBoostingClassifier(random_state=42)
boosted.fit(X_train, y_train)

pred_b = boosted.predict(X_test)
prob_b = boosted.predict_proba(X_test)[:, 1]

print("Always guess 'no diabetes':", round((1 - y_test.mean()) * 100, 1), "% correct")
print("Model accuracy:", round(accuracy_score(y_test, pred_b) * 100, 1), "%")
print("AUC:", round(roc_auc_score(y_test, prob_b), 3))
print("Precision:", round(precision_score(y_test, pred_b), 3))
print("Recall:", round(recall_score(y_test, pred_b), 3))

Always guess 'no diabetes': 85.3 % correct
Model accuracy: 85.3 %
AUC: 0.762
Precision: 0.483
Recall: 0.011


In [7]:
for name, probs in [("Logistic regression", prob), ("Gradient boosting", prob_b)]:
    print(name)
    for cutoff in [0.5, 0.3, 0.2, 0.15]:
        flagged = (probs >= cutoff).astype(int)
        p = precision_score(y_test, flagged, zero_division=0)
        r = recall_score(y_test, flagged, zero_division=0)
        print("  cutoff", cutoff, "| flagged:", flagged.sum(),
              "| precision:", round(p, 3), "| recall:", round(r, 3))
    print()

Logistic regression
  cutoff 0.5 | flagged: 819 | precision: 0.418 | recall: 0.043
  cutoff 0.3 | flagged: 5567 | precision: 0.373 | recall: 0.263
  cutoff 0.2 | flagged: 14173 | precision: 0.303 | recall: 0.544
  cutoff 0.15 | flagged: 21338 | precision: 0.265 | recall: 0.715

Gradient boosting
  cutoff 0.5 | flagged: 180 | precision: 0.483 | recall: 0.011
  cutoff 0.3 | flagged: 6761 | precision: 0.375 | recall: 0.321
  cutoff 0.2 | flagged: 16182 | precision: 0.299 | recall: 0.612
  cutoff 0.15 | flagged: 22826 | precision: 0.261 | recall: 0.753



In [8]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    boosted, X_test, y_test,
    scoring="roc_auc", n_repeats=5, random_state=42
)

for name, drop in sorted(zip(X.columns, result.importances_mean), key=lambda t: -t[1]):
    print(name, "| AUC drop when shuffled:", round(drop, 3))

age | AUC drop when shuffled: 0.146
bmi | AUC drop when shuffled: 0.071
income_level | AUC drop when shuffled: 0.021
exercise_yes | AUC drop when shuffled: 0.011
